# Case Ambev — Long Neck NENO
**Papel:** Especialista de Planejamento → responder ao VP de Vendas e VP de Logística

**Perguntas do case:**
1. Devemos seguir com os incentivos comerciais?
2. Qual o plano de produção e transferência?
3. Quanto vai custar?
4. Quais os riscos?

---
**Premissas:**
- AQ541 (Aquiraz/CE): cap. **12.600 hl/semana**
- NS541 (Pernambuco/PE): cap. **27.000 hl/semana**
- Cabotagem SP→NE: lead time **25 dias**
- Rodoviário: lead time **6 dias**, custo **+60%**, risco avaria **+5%**
- DOI mínimo NENO: **12 dias**
- BIAS histórico das GEOs: **+9%** (superestimam demanda)

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.float_format', '{:,.1f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 130)

semanas = ['W0_02fev', 'W1_09fev', 'W2_16fev', 'W3_23fev']
DOI_MIN = 12
CAP_AQ  = 12600
CAP_NS  = 27000

## 1. PCP — Produção Sugerida e Folgas Disponíveis

In [2]:
# AQ541 — Aquiraz/CE
pcp_aq = pd.DataFrame({
    'sku':      ['MALZBIER BRAHMA', 'PATAGONIA AMBER LAGER', 'COLORADO LAGER'],
    'W0_02fev': [0,     12240, 0],
    'W1_09fev': [9000,  1800,  0],
    'W2_16fev': [7560,  5040,  0],
    'W3_23fev': [0,     12600, 0],
})

# NS541 — Pernambuco/PE
pcp_ns = pd.DataFrame({
    'sku':      ['BRAHMA CHOPP ZERO', 'GOOSE ISLAND MIDWAY', 'MALZBIER BRAHMA',
                 'COLORADO LAGER', 'SKOL BEATS SENSES', 'BUDWEISER ZERO'],
    'W0_02fev': [0,    5400,  16200, 5400, 0,    0    ],
    'W1_09fev': [0,    14400, 0,     0,    0,    5400 ],
    'W2_16fev': [0,    0,     12960, 10800,3240, 0    ],
    'W3_23fev': [3600, 12600, 0,     0,    0,    10800],
})

# Folgas por semana
folgas = pd.DataFrame({'semana': semanas})
folgas['total_aq']    = [pcp_aq[s].sum()  for s in semanas]
folgas['folga_aq']    = CAP_AQ - folgas['total_aq']
folgas['total_ns']    = [pcp_ns[s].sum()  for s in semanas]
folgas['folga_ns']    = CAP_NS - folgas['total_ns']
folgas['folga_total'] = folgas['folga_aq'] + folgas['folga_ns']

print('=== AQ541 — Programação PCP ===')
display(pcp_aq)
print()
print('=== NS541 — Programação PCP ===')
display(pcp_ns)
print()
print('=== FOLGAS DISPONÍVEIS POR SEMANA (hl) ===')
display(folgas)
print(f"\n📌 Folga total: {folgas['folga_total'].sum():,.0f} hl")
print(f"   W0: {folgas.loc[0,'folga_total']:,.0f} hl (só AQ)")
print(f"   W1: {folgas.loc[1,'folga_total']:,.0f} hl (1.800 AQ + 7.200 NS) ← maior janela")
print(f"   W2 e W3: zero folga — linhas 100% ocupadas")

=== AQ541 — Programação PCP ===


,sku,W0_02fev,W1_09fev,W2_16fev,W3_23fev
0,MALZBIER BRAHMA,0,9000,7560,0
1,PATAGONIA AMBER LAGER,12240,1800,5040,12600
2,COLORADO LAGER,0,0,0,0



=== NS541 — Programação PCP ===


,sku,W0_02fev,W1_09fev,W2_16fev,W3_23fev
0,BRAHMA CHOPP ZERO,0,0,0,3600
1,GOOSE ISLAND MIDWAY,5400,14400,0,12600
2,MALZBIER BRAHMA,16200,0,12960,0
3,COLORADO LAGER,5400,0,10800,0
4,SKOL BEATS SENSES,0,0,3240,0
5,BUDWEISER ZERO,0,5400,0,10800



=== FOLGAS DISPONÍVEIS POR SEMANA (hl) ===


,semana,total_aq,folga_aq,total_ns,folga_ns,folga_total
0,W0_02fev,12240,360,27000,0,360
1,W1_09fev,10800,1800,19800,7200,9000
2,W2_16fev,12600,0,27000,0,0
3,W3_23fev,12600,0,27000,0,0



📌 Folga total: 9,360 hl
   W0: 360 hl (só AQ)
   W1: 9,000 hl (1.800 AQ + 7.200 NS) ← maior janela
   W2 e W3: zero folga — linhas 100% ocupadas


## 2. Demanda Malzbier — Nova Demanda (+30%)

In [3]:
malzbier = pd.DataFrame({
    'geo':      ['Mapapi', 'NE Norte', 'NE Sul', 'NO Araguaia', 'NO Centro'],
    'ei_w0':    [1985.6,   302.0,      4383.0,   0.0,           964.8     ],
    'W0_02fev': [6074.8,   1971.8,     3230.3,   78.9,          2227.5    ],
    'W1_09fev': [6286.4,   2265.8,     3517.6,   92.9,          2463.1    ],
    'W2_16fev': [4258.0,   1707.9,     2589.1,   62.9,          1789.6    ],
    'W3_23fev': [5204.6,   1844.0,     2857.3,   65.9,          2025.9    ],
})
malzbier['total_mes'] = malzbier[semanas].sum(axis=1)

dem_semana = [malzbier[s].sum() for s in semanas]
ei_total   = malzbier['ei_w0'].sum()
dem_total  = malzbier['total_mes'].sum()

display(malzbier)
print(f"\nDemanda total (nova, +30%):          {dem_total:>10,.1f} HL")
print(f"Demanda ajustada pelo BIAS (-9%):    {dem_total/1.09:>10,.1f} HL")
print(f"Estoque inicial total:               {ei_total:>10,.1f} HL")
print(f"\n🚨 NE Norte: EI = 302 HL — menos de 1 dia de estoque. Risco crítico de ruptura na W0!")

,geo,ei_w0,W0_02fev,W1_09fev,W2_16fev,W3_23fev,total_mes
0,Mapapi,"1,985.6","6,074.8","6,286.4","4,258.0","5,204.6","21,823.8"
1,NE Norte,302.0,"1,971.8","2,265.8","1,707.9","1,844.0","7,789.5"
2,NE Sul,"4,383.0","3,230.3","3,517.6","2,589.1","2,857.3","12,194.3"
3,NO Araguaia,0.0,78.9,92.9,62.9,65.9,300.6
4,NO Centro,964.8,"2,227.5","2,463.1","1,789.6","2,025.9","8,506.1"



Demanda total (nova, +30%):            50,614.3 HL
Demanda ajustada pelo BIAS (-9%):      46,435.1 HL
Estoque inicial total:                  7,635.4 HL

🚨 NE Norte: EI = 302 HL — menos de 1 dia de estoque. Risco crítico de ruptura na W0!


## 3. Balanço Semana a Semana — Sem Folgas vs Com Folgas
> DOI calculado sobre o total NENO. Mínimo exigido: **12 dias**

In [4]:
prod_pcp   = [
    pcp_aq.loc[pcp_aq['sku']=='MALZBIER BRAHMA', s].values[0] +
    pcp_ns.loc[pcp_ns['sku']=='MALZBIER BRAHMA', s].values[0]
    for s in semanas
]
folga_sem  = folgas['folga_total'].tolist()

# Demanda corrigida pelo BIAS de +9% (GEOs superestimam sistematicamente)
dem_semana_bias = [d / 1.09 for d in dem_semana]

resultados = []
for usar_folga in [False, True]:
    estoque = ei_total
    for i, s in enumerate(semanas):
        prod = prod_pcp[i] + (folga_sem[i] if usar_folga else 0)
        dem  = dem_semana_bias[i]  # demanda corrigida pelo BIAS
        dem_prox = dem_semana_bias[i+1] if i < 3 else dem_semana_bias[i]  # DOI prospectivo
        ef   = estoque + prod - dem
        doi  = ef / dem_prox * 6  # DOI prospectivo: usa demanda da semana seguinte
        resultados.append({
            'cenario':        'COM folgas' if usar_folga else 'SEM folgas',
            'semana':         s,
            'EI (hl)':        round(estoque, 0),
            'prod malzbier':  round(prod, 0),
            'demanda c/BIAS': round(dem, 0),
            'EF (hl)':        round(ef, 0),
            'DOI (dias)':     round(doi, 1),
            'status':         '✅' if doi >= DOI_MIN else '🚨'
        })
        estoque = ef  # valor exato, sem arredondamento

df_res = pd.DataFrame(resultados)

print('─── SEM FOLGAS (PCP original) ───')
display(df_res[df_res['cenario']=='SEM folgas'].drop(columns='cenario').reset_index(drop=True))

print()
print('─── COM FOLGAS (toda folga → Malzbier) ───')
display(df_res[df_res['cenario']=='COM folgas'].drop(columns='cenario').reset_index(drop=True))

print()
print('📌 CONCLUSÃO:')
print(f'   Mesmo usando 100% das folgas ({sum(folga_sem):,.0f} hl), o DOI de 12d')
print(f'   NÃO é atingido em W0 (5,6d), W1 (7,3d) e W3 (8,9d).')
print(f'   Pior semana: W3 — zero produção e zero folga → transferência de SP obrigatória.')

─── SEM FOLGAS (PCP original) ───


,semana,EI (hl),prod malzbier,demanda c/BIAS,EF (hl),DOI (dias),status
0,W0_02fev,"7,635.0",16200,"12,462.0","11,374.0",5.1,🚨
1,W1_09fev,"11,374.0",9000,"13,418.0","6,955.0",4.4,🚨
2,W2_16fev,"6,955.0",20520,"9,548.0","17,927.0",9.8,🚨
3,W3_23fev,"17,927.0",0,"11,007.0","6,920.0",3.8,🚨



─── COM FOLGAS (toda folga → Malzbier) ───


,semana,EI (hl),prod malzbier,demanda c/BIAS,EF (hl),DOI (dias),status
0,W0_02fev,"7,635.0",16560,"12,462.0","11,734.0",5.2,🚨
1,W1_09fev,"11,734.0",18000,"13,418.0","16,315.0",10.3,🚨
2,W2_16fev,"16,315.0",20520,"9,548.0","27,287.0",14.9,✅
3,W3_23fev,"27,287.0",0,"11,007.0","16,280.0",8.9,🚨



📌 CONCLUSÃO:
   Mesmo usando 100% das folgas (9,360 hl), o DOI de 12d
   NÃO é atingido em W0 (5,6d), W1 (7,3d) e W3 (8,9d).
   Pior semana: W3 — zero produção e zero folga → transferência de SP obrigatória.


## 4. Volume de Transferência Necessário de SP (para garantir DOI ≥ 12d)

In [5]:
print('=== VOLUME DE TRANSFERÊNCIA NECESSÁRIO ===')
print('(com folgas já aproveitadas + transferência para atingir DOI >= 12d)\n')

estoque = ei_total
transf_necessaria = []
rows = []
for i, s in enumerate(semanas):
    prod          = prod_pcp[i] + folga_sem[i]
    dem           = dem_semana_bias[i]  # demanda corrigida pelo BIAS
    dem_prox      = dem_semana_bias[i+1] if i < 3 else dem_semana_bias[i]  # DOI prospectivo
    ef_sem_transf = estoque + prod - dem
    ef_min        = DOI_MIN * dem_prox / 6  # ef mínimo baseado na demanda da semana seguinte
    transf        = max(0, ef_min - ef_sem_transf)
    transf_embarque = transf / 0.95  # +5% avaria: embarca mais para chegar o volume necessário
    ef_final      = ef_sem_transf + transf
    doi_final     = ef_final / dem_prox * 6  # DOI prospectivo
    transf_necessaria.append(transf_embarque)
    rows.append({
        'semana':              s,
        'EF s/ transf':        round(ef_sem_transf, 0),
        'EF mínimo (12d)':     round(ef_min, 0),
        'Transf. chegada (hl)':round(transf, 0),
        'Embarque rodo (hl)':  round(transf_embarque, 0),
        'EF final':            round(ef_final, 0),
        'DOI final (d)':       round(doi_final, 1),
    })
    estoque = ef_final  # valor exato, sem arredondamento

df_transf = pd.DataFrame(rows)
display(df_transf)

total_transf = sum(transf_necessaria)
print(f'\nVOLUME TOTAL A TRANSFERIR DE SP: {total_transf:,.0f} HL')
print(f'\n⚠️  Cabotagem (lead 25d) já está fora do prazo para Fevereiro.')
print(f'   Única opção viável: RODOVIÁRIO (lead 6d) — mas com custo e risco maiores.')

=== VOLUME DE TRANSFERÊNCIA NECESSÁRIO ===
(com folgas já aproveitadas + transferência para atingir DOI >= 12d)



,semana,EF s/ transf,EF mínimo (12d),Transf. chegada (hl),Embarque rodo (hl),EF final,DOI final (d)
0,W0_02fev,"11,734.0","26,836.0","15,103.0","15,898.0","26,836.0",12.0
1,W1_09fev,"31,418.0","19,096.0",0.0,0.0,"31,418.0",19.7
2,W2_16fev,"42,390.0","22,014.0",0.0,0.0,"42,390.0",23.1
3,W3_23fev,"31,383.0","22,014.0",0.0,0.0,"31,383.0",17.1



VOLUME TOTAL A TRANSFERIR DE SP: 15,898 HL

⚠️  Cabotagem (lead 25d) já está fora do prazo para Fevereiro.
   Única opção viável: RODOVIÁRIO (lead 6d) — mas com custo e risco maiores.


## 5. Custos — Produção Local vs Transferência

In [6]:
cabo_ba  = 85  # pressupondo que o custo de transf do excel é o de cabo
cabo_pb  = 95
rodo_ba  = cabo_ba * 1.6
rodo_pb  = cabo_pb * 1.6
avaria   = 1.05
maco     = 285

cabo_medio = (cabo_ba + cabo_pb) / 2  # média dos cabos/rodos para efeito de simplificação
rodo_medio = (rodo_ba + rodo_pb) / 2  # avaria já dentro do volume de embarque 

custo_rodo = total_transf * rodo_medio

maco_local = maco
maco_rodo  = maco - rodo_medio  # lembrar que maco não desconta logística (e pro case a interna é 0)

print('=== CUSTO DE TRANSFERÊNCIA ===')
print(f'  Volume a transferir de SP:         {total_transf:>10,.0f} HL')
print()
print(f'  Cabotagem → BA:   R$ {cabo_ba:.2f}/HL  |  → PB: R$ {cabo_pb:.2f}/HL  |  Médio: R$ {cabo_medio:.2f}/HL')
print('   Cabotagem é inviável para o prazo de fevereiro')
print(f'  Rodoviário → BA:  R$ {rodo_ba:.2f}/HL  |  → PB: R$ {rodo_pb:.2f}/HL  |  Médio: R$ {rodo_medio:.2f}/HL')
print()
print(f'  Custo total rodoviário:            R$ {custo_rodo:>12,.0f}')
print()
print('=== MACO LÍQUIDO (após frete) ===')
print(f'  Produção local NENO:               R$ {maco_local:.2f}/HL ← melhor cenário')
print(f'  Transferência via rodoviário:      R$ {maco_rodo:.2f}/HL {"🚨 MACO NEGATIVO" if maco_rodo < 0 else "⚠️  margem comprimida"}')

=== CUSTO DE TRANSFERÊNCIA ===
  Volume a transferir de SP:             15,898 HL

  Cabotagem → BA:   R$ 85.00/HL  |  → PB: R$ 95.00/HL  |  Médio: R$ 90.00/HL
   Cabotagem é inviável para o prazo de fevereiro
  Rodoviário → BA:  R$ 136.00/HL  |  → PB: R$ 152.00/HL  |  Médio: R$ 144.00/HL

  Custo total rodoviário:            R$    2,289,247

=== MACO LÍQUIDO (após frete) ===
  Produção local NENO:               R$ 285.00/HL ← melhor cenário
  Transferência via rodoviário:      R$ 141.00/HL ⚠️  margem comprimida


## 6. Recomendação Final

In [7]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║         RECOMENDAÇÃO — ESPECIALISTA DE PLANEJAMENTO             ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. INCENTIVOS COMERCIAIS (Malzbier +30%)?                       ║
║     → Aceitar COM CAUTELA                                        ║
║     → BIAS +9%: demanda real provavelmente ~46.800 HL           ║
║     → Condicionar incentivo à realização semana a semana        ║
║                                                                  ║
║  2. PLANO DE PRODUÇÃO                                            ║
║     → Usar toda folga disponível para Malzbier:                  ║
║       W0: +360 hl (AQ541)                                        ║
║       W1: +9.000 hl (1.800 AQ + 7.200 NS)  ← prioridade        ║
║     → Mesmo assim DOI < 12d em W0, W1 e W3                      ║
║     → Logo, transferência de SP necessária para W3               ║
║                                                                  ║
║  3. MODAL DE TRANSFERÊNCIA                                       ║
║     → Cabotagem ATRASADA (lead 25d, já fora do prazo)           ║
║     → Rodoviário é o único modal viável para Fev (lead 6d)      ║
║     → Volume mínimo necessário via rodo para garantir DOI 12d   ║
║     → Rodoviário corrói margem — avaliar se vale vs ruptura.    ║
      → Custo seria de 2 milhões                                  ║
║                                                                  ║
║  4. RISCOS                                                       ║
║     🚨 NE Norte: EI = 302 HL → ruptura iminente na W0           ║
║     🚨 W3: zero produção + zero folga → semana crítica          ║
║     🚨 Rodoviário: +5% avaria + MACO comprimido                 ║
║     🚨 BIAS +9%: risco de overstock se demanda não realizar     ║
      (mesmo com correção)                                        ║
║     🚨 Crescimento +10%/mês a partir de mar: avaliar nova linha ║
╚══════════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════════╗
║         RECOMENDAÇÃO — ESPECIALISTA DE PLANEJAMENTO             ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. INCENTIVOS COMERCIAIS (Malzbier +30%)?                       ║
║     → Aceitar COM CAUTELA                                        ║
║     → BIAS +9%: demanda real provavelmente ~46.800 HL           ║
║     → Condicionar incentivo à realização semana a semana        ║
║                                                                  ║
║  2. PLANO DE PRODUÇÃO                                            ║
║     → Usar toda folga disponível para Malzbier:                  ║
║       W0: +360 hl (AQ541)                                        ║
║       W1: +9.000 hl (1.800 AQ + 7.200 NS)  ← prioridade        ║
║     → Mesmo assim DOI < 12d em W0, W1 e W3                      ║
║     → Logo, transferência de SP neces